# Regime detection on satellite revisit series

Quick scratch notebook to verify the HMM module behaves the way I want
on a synthetic revisit series with a planted regime shift.  This is the
same kind of plot I'd make for a vol-regime model on intraday returns -
the workflow really does port over more or less unchanged.

What I'm checking:

1. Baum-Welch identifies the two regime means within ~0.05 of truth.
2. Viterbi path lights up the planted change point.
3. BOCPD's CP posterior spikes near the same point with no info from HMM.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'quant' / 'src'))

import numpy as np
import matplotlib.pyplot as plt

from geoalpha_quant.regime import GaussianHMM, BayesianOnlineChangePoint
from geoalpha_quant.io import make_synthetic_revisit_series

series, truth = make_synthetic_revisit_series(n_obs=320, seed=4, regime_shift_at=160)
print('series', series.shape, 'shift_at', np.where(np.diff(truth))[0])

In [ ]:
hmm = GaussianHMM(n_states=2, seed=1).fit(series)
path = hmm.predict(series)
print('mu  ', hmm.params.mu)
print('var ', hmm.params.var)
print('log-lik', hmm.score(series))

In [ ]:
bocpd = BayesianOnlineChangePoint(hazard_lambda=80.0)
cp = bocpd.run(series)
print('peak BOCPD CP prob @', cp.argmax(), 'val', cp.max().round(3))

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax1.plot(series, color='tab:gray', lw=1.0, label='NDVI proxy')
ax1.scatter(np.arange(len(series)), series, c=path, cmap='coolwarm', s=8, label='HMM state')
ax1.axvline(160, ls='--', color='k', alpha=0.4)
ax1.set_ylabel('NDVI proxy'); ax1.legend(loc='upper right')
ax2.plot(cp, color='tab:purple', label='BOCPD P(change)')
ax2.axvline(160, ls='--', color='k', alpha=0.4)
ax2.set_ylabel('CP posterior'); ax2.set_xlabel('revisit #'); ax2.legend()
plt.tight_layout()
plt.show()

Both detectors agree on the planted shift, which is what I wanted to
confirm before wiring the API endpoints up.  Next step: re-run on a
real Sentinel-2 NDVI extract and compare the BOCPD posterior to the
ground-truth deforestation polygons.
